# Social Progress Analysis

Exploratory analysis of World Happiness Report data (2011–2025) used by the Social Progress Analytics dashboard.

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import plotly.express as px

ROOT = Path.cwd().parent if Path.cwd().name == "notebook" else Path.cwd()
sys.path.insert(0, str(ROOT))

from utils.data_loader import load_data
from utils.config import FACTOR_COLUMNS

DATA_PATH = ROOT / "data" / "WHR26_Data_Figure_2.1.xlsx"

In [ ]:
df = load_data(DATA_PATH)
print(f"Rows: {len(df):,} | Years: {df['Year'].min():.0f}–{df['Year'].max():.0f} | Countries: {df['Country name'].nunique()}")
df.head()

## Global life evaluation trend

In [ ]:
yearly = df.groupby("Year")["Life Evaluation"].mean().reset_index()

fig = px.line(
    yearly,
    x="Year",
    y="Life Evaluation",
    markers=True,
    title="Average global life evaluation score by year",
)
fig.update_layout(yaxis_title="Life evaluation (0–10)")
fig.show()

## Top 15 countries (latest year)

In [ ]:
latest_year = df["Year"].max()
latest = df[df["Year"] == latest_year].sort_values("Rank").head(15)

fig = px.bar(
    latest.sort_values("Life Evaluation"),
    x="Life Evaluation",
    y="Country name",
    orientation="h",
    title=f"Top 15 countries — {int(latest_year)}",
    color_discrete_sequence=["#3b82f6"],
)
fig.show()

## Factor correlation with life evaluation

In [ ]:
factor_cols = list(FACTOR_COLUMNS.values())
corr = (
    df[["Life Evaluation"] + factor_cols]
    .corr()["Life Evaluation"]
    .drop("Life Evaluation")
    .sort_values(ascending=True)
    .reset_index()
)
corr.columns = ["Factor", "Correlation"]

fig = px.bar(
    corr,
    x="Correlation",
    y="Factor",
    orientation="h",
    title="Correlation of explanatory factors with life evaluation",
    color_discrete_sequence=["#14b8a6"],
)
fig.show()